In [3]:
# Install PySpark 
!pip install pyspark

# Import libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count

In [5]:
spark = SparkSession.builder \
    .appName("Coffee Sales Big Data Analysis") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 10:27:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/anupamacodtech/coffee-sales/Coffe_sales.csv


In [9]:
# Update path if needed
df = spark.read.csv('/kaggle/input/datasets/anupamacodtech/coffee-sales/Coffe_sales.csv', 
                    header=True, inferSchema=True)

df.show(5)
df.printSchema()

+-----------+---------+-----+-------------+-----------+-------+----------+-----------+---------+----------+--------------------+
|hour_of_day|cash_type|money|  coffee_name|Time_of_Day|Weekday|Month_name|Weekdaysort|Monthsort|      Date|                Time|
+-----------+---------+-----+-------------+-----------+-------+----------+-----------+---------+----------+--------------------+
|         10|     card| 38.7|        Latte|    Morning|    Fri|       Mar|          5|        3|2024-03-01|2026-05-03 10:15:...|
|         12|     card| 38.7|Hot Chocolate|  Afternoon|    Fri|       Mar|          5|        3|2024-03-01|2026-05-03 12:19:...|
|         12|     card| 38.7|Hot Chocolate|  Afternoon|    Fri|       Mar|          5|        3|2024-03-01|2026-05-03 12:20:...|
|         13|     card| 28.9|    Americano|  Afternoon|    Fri|       Mar|          5|        3|2024-03-01|2026-05-03 13:46:...|
|         13|     card| 38.7|        Latte|  Afternoon|    Fri|       Mar|          5|        3|2

In [16]:
# Drop null values
df = df.dropna()

# Remove duplicates
df = df.dropDuplicates()

print("Total records:", df.count())

Total records: 3547


In [20]:
from pyspark.sql.functions import sum, col

total_sales = df.select(sum(col("total_sales"))).collect()[0][0]
print("Total Sales:", total_sales)

Total Sales: 112245.5800000002


In [21]:
df.columns

['hour_of_day',
 'cash_type',
 'price',
 'coffee_name',
 'Time_of_Day',
 'Weekday',
 'Month_name',
 'Weekdaysort',
 'Monthsort',
 'Date',
 'Time',
 'total_sales']

In [25]:
# Feature Engineering
# Rename column for clarity
df = df.withColumnRenamed("money", "price")

# Create total sales column
df = df.withColumn("total_sales", col("price"))
print(df)

DataFrame[hour_of_day: int, cash_type: string, price: double, coffee_name: string, Time_of_Day: string, Weekday: string, Month_name: string, Weekdaysort: int, Monthsort: int, Date: date, Time: timestamp, total_sales: double]


In [27]:
from pyspark.sql.functions import sum, col

top_products = df.groupBy("coffee_name").agg(
    sum("total_sales").alias("Total_Revenue")
).orderBy(col("Total_Revenue").desc())

top_products.show()


+-------------------+------------------+
|        coffee_name|     Total_Revenue|
+-------------------+------------------+
|              Latte|  26875.2999999998|
|Americano with Milk|24751.119999999948|
|         Cappuccino|17439.139999999996|
|          Americano|14650.259999999886|
|      Hot Chocolate|  9933.46000000003|
|              Cocoa| 8521.160000000027|
|            Cortado| 7384.860000000018|
|           Espresso| 2690.279999999995|
+-------------------+------------------+



In [29]:
df.groupBy("Weekday").agg(
    sum("total_sales").alias("Revenue")
).show()

+-------+------------------+
|Weekday|           Revenue|
+-------+------------------+
|    Sun|13336.060000000009|
|    Mon| 17363.09999999998|
|    Thu|16091.400000000005|
|    Sat| 14733.51999999999|
|    Wed|15750.460000000006|
|    Fri| 16802.65999999999|
|    Tue|18168.379999999983|
+-------+------------------+



In [34]:
# Sales by Coffee Type
coffee_sales = df.groupBy("coffee_name") \
    .agg(sum("total_sales").alias("revenue")) \
    .orderBy(col("revenue").desc())

coffee_sales.show()

+-------------------+------------------+
|        coffee_name|           revenue|
+-------------------+------------------+
|              Latte|  26875.2999999998|
|Americano with Milk|24751.119999999948|
|         Cappuccino|17439.139999999996|
|          Americano|14650.259999999886|
|      Hot Chocolate|  9933.46000000003|
|              Cocoa| 8521.160000000027|
|            Cortado| 7384.860000000018|
|           Espresso| 2690.279999999995|
+-------------------+------------------+



In [35]:
# Sales by Time of Day
time_sales = df.groupBy("Time_of_Day") \
    .agg(sum("total_sales").alias("revenue")) \
    .orderBy(col("revenue").desc())

time_sales.show()

+-----------+-----------------+
|Time_of_Day|          revenue|
+-----------+-----------------+
|      Night|38186.33999999978|
|  Afternoon|38130.03999999974|
|    Morning| 35929.1999999997|
+-----------+-----------------+



In [36]:
# Sales by Payment Type
payment_sales = df.groupBy("cash_type") \
    .agg(sum("total_sales").alias("revenue"))

payment_sales.show()

+---------+-----------------+
|cash_type|          revenue|
+---------+-----------------+
|     card|112245.5800000002|
+---------+-----------------+



In [38]:
# Peak Sales Hour
hour_sales = df.groupBy("hour_of_day") \
    .agg(sum("total_sales").alias("revenue")) \
    .orderBy(col("revenue").desc())

hour_sales.show()

+-----------+------------------+
|hour_of_day|           revenue|
+-----------+------------------+
|         10|10198.520000000004|
|         16| 9031.840000000017|
|         11|  8453.10000000001|
|         19| 7751.960000000008|
|         17| 7659.760000000007|
|         15|  7476.02000000001|
|         12| 7419.620000000006|
|          9| 7264.280000000005|
|         14| 7173.800000000006|
|         18| 7162.600000000007|
|         13|7028.7600000000075|
|          8| 7017.880000000009|
|         21|6397.9400000000105|
|         20| 5578.920000000006|
|         22| 3635.160000000005|
|          7|2846.0200000000027|
|          6|             149.4|
+-----------+------------------+



In [39]:
# Monthly Sales Trend
monthly_sales = df.groupBy("Month_name") \
    .agg(sum("total_sales").alias("revenue")) \
    .orderBy(col("revenue").desc())

monthly_sales.show()

+----------+------------------+
|Month_name|           revenue|
+----------+------------------+
|       Mar|15891.639999999994|
|       Oct|13891.160000000029|
|       Feb|13215.480000000003|
|       Sep| 9988.640000000007|
|       Nov| 8590.540000000019|
|       Dec| 8237.740000000018|
|       May| 8164.420000000008|
|       Jun| 7617.760000000004|
|       Aug| 7613.840000000007|
|       Jul|6915.9400000000005|
|       Jan| 6398.860000000011|
|       Apr| 5719.559999999994|
+----------+------------------+



In [40]:
#Scalability Demonstration
# Check partitions
print("Before:", df.rdd.getNumPartitions())

# Increase partitions (simulate large data handling)
df = df.repartition(4)

print("After:", df.rdd.getNumPartitions())

Before: 1
After: 4
